In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc

In [2]:
df = pd.read_csv("/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_positive_reg_pi_on_r_add_scaled.csv", index_col=0)
df.columns

Index(['k_on', 'k_off', 'mrna_half_life', 'protein_half_life', 'k_prod_mRNA',
       'k_prod_protein', 'pi_on', 'n_gene_1_to_gene_2',
       'k_add_gene_1_to_gene_2', 'n_gene_2_to_gene_1',
       'k_add_gene_2_to_gene_1', 'n_gene_1_to_gene_3',
       'k_add_gene_1_to_gene_3', 'n_gene_2_to_gene_3',
       'k_add_gene_2_to_gene_3', 'pair_id', 'gene_id'],
      dtype='object')

In [3]:
set_1_ids = set(df[(df['k_off'] < 0.06) & (df['gene_id'] != 3)]['pair_id'])
set_2_ids = set(df[(df['k_off'] > 55) & (df['gene_id'] != 3)]['pair_id'])

union_size = len(set_1_ids | set_2_ids)  # union of sets
union_size

8986

In [7]:
df_filtered = df[(df['k_off'] > 0.06) & (df['k_off'] < 55)]
# group by pair_id and collect which gene_ids are present
pair_gene_sets = df_filtered.groupby("pair_id")["gene_id"].apply(set)

# keep only pair_ids that have both gene 1 and 2
pair_ids_with_1_and_2 = pair_gene_sets[pair_gene_sets.apply(lambda x: {1, 2}.issubset(x))].index

# sample 1000 of them (or all if fewer than 1000)
sampled_pair_ids = pd.Series(pair_ids_with_1_and_2).sample(n=1000, random_state=42)

# filter your df for those sampled pair_ids
df_sampled = df_filtered[df_filtered["pair_id"].isin(sampled_pair_ids)]
df_sampled = df_sampled[df_sampled['gene_id'] < 3]
df_sampled

,k_on,k_off,mrna_half_life,protein_half_life,k_prod_mRNA,k_prod_protein,pi_on,n_gene_1_to_gene_2,k_add_gene_1_to_gene_2,n_gene_2_to_gene_1,k_add_gene_2_to_gene_1,n_gene_1_to_gene_3,k_add_gene_1_to_gene_3,n_gene_2_to_gene_3,k_add_gene_2_to_gene_3,pair_id,gene_id
24,0.268995,0.841603,2.684457,52.055736,9.905942,64.169898,0.242207,0.559347,0.069839,0.465577,0.172111,1.397771,4.132764,0.300907,3.054073,8,1
25,0.033092,0.449836,1.003879,30.182881,0.967315,972.017300,0.068524,0.559347,0.069839,0.465577,0.172111,1.397771,4.132764,0.300907,3.054073,8,2
90,0.095468,2.503241,9.887630,9.746546,23.096406,36.291064,0.036737,0.268099,0.188465,4.559236,0.098977,1.563060,7.629753,3.493557,1.545001,30,1
91,0.030906,3.219517,8.487686,15.382692,0.727626,1046.357201,0.009508,0.268099,0.188465,4.559236,0.098977,1.563060,7.629753,3.493557,1.545001,30,2
207,0.468425,5.163922,4.555133,14.211248,9.843271,320.210342,0.083167,0.497818,2.137316,0.370939,4.557163,0.525431,1.037021,0.370031,0.670907,69,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74809,0.173184,14.091477,0.665271,106.992161,31.657376,395.660551,0.012141,0.420854,0.481636,2.641533,0.967850,0.588971,0.044785,0.306291,0.199395,24936,2
74886,0.330459,2.892827,11.689999,14.903789,1.403693,357.578050,0.102522,0.971161,0.028548,0.626975,0.728813,0.147933,0.582623,0.562275,0.073864,24962,1
74887,0.050962,4.699919,2.612512,9.072920,0.459181,75.442130,0.010727,0.971161,0.028548,0.626975,0.728813,0.147933,0.582623,0.562275,0.073864,24962,2
74958,0.022940,1.141056,1.516216,12.013994,7.673699,50.972896,0.019708,0.146787,0.026722,1.895170,0.119041,2.561765,0.018439,1.105825,0.013307,24986,1


In [9]:
# Step 1: compute only for gene_id = 2
df_sampled['k_add_gene_1_to_gene_2'] = pd.NA
df_sampled.loc[df_sampled['gene_id'] == 2, 'k_add_gene_1_to_gene_2'] = (
    10 * df_sampled.loc[df_sampled['gene_id'] == 2, 'k_on']
)

# Step 2: copy that value specifically to gene_id = 1 rows in the same pair_id
map_vals = df_sampled.loc[df_sampled['gene_id'] == 2, ['pair_id', 'k_add_gene_1_to_gene_2']]
df_sampled = df_sampled.merge(map_vals, on='pair_id', how='left', suffixes=('', '_from_gene2'))

# Now fill NaN (like for gene_id=1 row) with the value from its gene2 partner
df_sampled['k_add_gene_1_to_gene_2'] = df_sampled['k_add_gene_1_to_gene_2'].fillna(
    df_sampled['k_add_gene_1_to_gene_2_from_gene2']
)

# cleanup
df_sampled = df_sampled.drop(columns=['k_add_gene_1_to_gene_2_from_gene2'])
df_sampled

/tmp/ipykernel_3000522/308990437.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sampled['k_add_gene_1_to_gene_2'] = df_sampled['k_add_gene_1_to_gene_2'].fillna(


,k_on,k_off,mrna_half_life,protein_half_life,k_prod_mRNA,k_prod_protein,pi_on,n_gene_1_to_gene_2,k_add_gene_1_to_gene_2,n_gene_2_to_gene_1,k_add_gene_2_to_gene_1,n_gene_1_to_gene_3,k_add_gene_1_to_gene_3,n_gene_2_to_gene_3,k_add_gene_2_to_gene_3,pair_id,gene_id
0,0.268995,0.841603,2.684457,52.055736,9.905942,64.169898,0.242207,0.559347,0.330922,0.465577,0.172111,1.397771,4.132764,0.300907,3.054073,8,1
1,0.033092,0.449836,1.003879,30.182881,0.967315,972.017300,0.068524,0.559347,0.330922,0.465577,0.172111,1.397771,4.132764,0.300907,3.054073,8,2
2,0.095468,2.503241,9.887630,9.746546,23.096406,36.291064,0.036737,0.268099,0.309060,4.559236,0.098977,1.563060,7.629753,3.493557,1.545001,30,1
3,0.030906,3.219517,8.487686,15.382692,0.727626,1046.357201,0.009508,0.268099,0.309060,4.559236,0.098977,1.563060,7.629753,3.493557,1.545001,30,2
4,0.468425,5.163922,4.555133,14.211248,9.843271,320.210342,0.083167,0.497818,16.825906,0.370939,4.557163,0.525431,1.037021,0.370031,0.670907,69,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,0.173184,14.091477,0.665271,106.992161,31.657376,395.660551,0.012141,0.420854,1.731843,2.641533,0.967850,0.588971,0.044785,0.306291,0.199395,24936,2
1996,0.330459,2.892827,11.689999,14.903789,1.403693,357.578050,0.102522,0.971161,0.509616,0.626975,0.728813,0.147933,0.582623,0.562275,0.073864,24962,1
1997,0.050962,4.699919,2.612512,9.072920,0.459181,75.442130,0.010727,0.971161,0.509616,0.626975,0.728813,0.147933,0.582623,0.562275,0.073864,24962,2
1998,0.022940,1.141056,1.516216,12.013994,7.673699,50.972896,0.019708,0.146787,0.167160,1.895170,0.119041,2.561765,0.018439,1.105825,0.013307,24986,1


In [ ]:
df_sampled['pair_id'] = df_sampled.index//2
df_sampled

In [14]:
df_sampled.to_csv("/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_2genes_positive_k_add_10_k_on.csv")

,k_on,k_off,mrna_half_life,protein_half_life,k_prod_mRNA,k_prod_protein,pi_on,n_gene_1_to_gene_2,k_add_gene_1_to_gene_2,n_gene_2_to_gene_1,k_add_gene_2_to_gene_1,n_gene_1_to_gene_3,k_add_gene_1_to_gene_3,n_gene_2_to_gene_3,k_add_gene_2_to_gene_3,pair_id,gene_id
0,0.268995,0.841603,2.684457,52.055736,9.905942,64.169898,0.242207,0.559347,0.330922,0.465577,0.172111,1.397771,4.132764,0.300907,3.054073,0,1
1,0.033092,0.449836,1.003879,30.182881,0.967315,972.017300,0.068524,0.559347,0.330922,0.465577,0.172111,1.397771,4.132764,0.300907,3.054073,0,2
2,0.095468,2.503241,9.887630,9.746546,23.096406,36.291064,0.036737,0.268099,0.309060,4.559236,0.098977,1.563060,7.629753,3.493557,1.545001,1,1
3,0.030906,3.219517,8.487686,15.382692,0.727626,1046.357201,0.009508,0.268099,0.309060,4.559236,0.098977,1.563060,7.629753,3.493557,1.545001,1,2
4,0.468425,5.163922,4.555133,14.211248,9.843271,320.210342,0.083167,0.497818,16.825906,0.370939,4.557163,0.525431,1.037021,0.370031,0.670907,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,0.173184,14.091477,0.665271,106.992161,31.657376,395.660551,0.012141,0.420854,1.731843,2.641533,0.967850,0.588971,0.044785,0.306291,0.199395,997,2
1996,0.330459,2.892827,11.689999,14.903789,1.403693,357.578050,0.102522,0.971161,0.509616,0.626975,0.728813,0.147933,0.582623,0.562275,0.073864,998,1
1997,0.050962,4.699919,2.612512,9.072920,0.459181,75.442130,0.010727,0.971161,0.509616,0.626975,0.728813,0.147933,0.582623,0.562275,0.073864,998,2
1998,0.022940,1.141056,1.516216,12.013994,7.673699,50.972896,0.019708,0.146787,0.167160,1.895170,0.119041,2.561765,0.018439,1.105825,0.013307,999,1


# Parameter generation


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc

# --- Define sampled parameters ---
gene_params_sampled = [
    "pi_on",            # k_on / (k_on + k_off)
    "k_on",             # directly sampled k_on
    "mrna_half_life",
    "protein_half_life",
    "k_prod_protein",
    "k_prod_mRNA"
]

interaction_params_scaled = [
    "n_gene_1_to_gene_2", "k_add_scaled_gene_1_to_gene_2",
    "n_gene_2_to_gene_1", "k_add_scaled_gene_2_to_gene_1",
    "n_gene_1_to_gene_3", "k_add_scaled_gene_1_to_gene_3",
    "n_gene_2_to_gene_3", "k_add_scaled_gene_2_to_gene_3"
]

param_names = (
    [f"{p}_gene_1" for p in gene_params_sampled] +
    [f"{p}_gene_2" for p in gene_params_sampled] +
    [f"{p}_gene_3" for p in gene_params_sampled] +
    interaction_params_scaled
)

# --- Bounds for sampled parameters ---
param_bounds = {
    # Gene-level
    "pi_on": (0.002, 0.4),                  # activation probability
    "k_on": (0.01, 3),                      # directly sampled
    "mrna_half_life": (0.6, 17),
    "protein_half_life": (7, 200),
    "k_prod_mRNA": (0.2, 60),
    "k_prod_protein": (19, 2700),
    # Interaction-level
    "n_gene_1_to_gene_2": (0.1, 5),
    "n_gene_2_to_gene_1": (0.1, 5),
    "n_gene_1_to_gene_3": (0.1, 5),
    "n_gene_2_to_gene_3": (0.1, 5),
    "k_add_scaled_gene_1_to_gene_2": (0.5, 10),
    "k_add_scaled_gene_2_to_gene_1": (0.5, 10),
    "k_add_scaled_gene_1_to_gene_3": (0.5, 10),
    "k_add_scaled_gene_2_to_gene_3": (0.5, 10),
}

bounds = (
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in interaction_params_scaled]
)

# --- Sampling configuration ---
n_valid_required = 25000
seed = 42

# Latin Hypercube Sampling (log10 for all except pi_on)
log_bounds_lower = [
    np.log10(b[0]) if "pi_on" not in param_names[i] else b[0]
    for i, b in enumerate(bounds)
]
log_bounds_upper = [
    np.log10(b[1]) if "pi_on" not in param_names[i] else b[1]
    for i, b in enumerate(bounds)
]

sampler = qmc.LatinHypercube(d=len(bounds), seed=seed)
sample = sampler.random(n=n_valid_required)

scaled_sample = np.empty_like(sample)
for i, name in enumerate(param_names):
    # Determine log bounds for every parameter (including pi_on)
    lower = np.log10(param_bounds["pi_on"][0]) if "pi_on" in name else log_bounds_lower[i]
    upper = np.log10(param_bounds["pi_on"][1]) if "pi_on" in name else log_bounds_upper[i]

    # Scale in log-space and exponentiate back
    scaled_log = qmc.scale(sample[:, [i]], [lower], [upper]).ravel()
    scaled_sample[:, i] = 10 ** scaled_log



df_sampled = pd.DataFrame(scaled_sample, columns=param_names)

# --- Convert to actual k_off and k_add ---
def convert_params(row):
    converted = {}

    for g in [1, 2, 3]:
        pi = row[f"pi_on_gene_{g}"]
        k_on = row[f"k_on_gene_{g}"]

        # Derive k_off from pi_on and k_on
        k_off = k_on * (1 - pi) / pi

        converted[f"k_on_gene_{g}"] = k_on
        converted[f"k_off_gene_{g}"] = k_off
        converted[f"mrna_half_life_gene_{g}"] = row[f"mrna_half_life_gene_{g}"]
        converted[f"protein_half_life_gene_{g}"] = row[f"protein_half_life_gene_{g}"]
        converted[f"k_prod_mRNA_gene_{g}"] = row[f"k_prod_mRNA_gene_{g}"]
        converted[f"k_prod_protein_gene_{g}"] = row[f"k_prod_protein_gene_{g}"]

    # Convert k_add_scaled → k_add
    for key in row.index:
        if "k_add_scaled" in key:
            tgt_gene = key.split("_")[7]  # target gene index
            k_on_target = converted[f"k_on_gene_{tgt_gene}"]
            converted[key.replace("k_add_scaled", "k_add")] = row[key] * k_on_target

        elif "n_gene" in key:
            converted[key] = row[key]

    return pd.Series(converted)

df_converted = df_sampled.apply(convert_params, axis=1)

# --- Expand to long format ---
rows = []
for idx, row in df_converted.iterrows():
    g1 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_1") and "to" not in k}
    g2 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_2") and "to" not in k}
    g3 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_3") and "to" not in k}
    interactions = {k: v for k, v in row.items() if "gene_" in k and "to" in k}

    rows.append({**g1, **interactions, "pair_id": idx, "gene_id": 1})
    rows.append({**g2, **interactions, "pair_id": idx, "gene_id": 2})
    rows.append({**g3, **interactions, "pair_id": idx, "gene_id": 3})

final_df = pd.DataFrame(rows).reset_index(drop=True)

# --- Save ---
output_path = (
    "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_positive_reg_pi_on_r_add_scaled.csv"
)
final_df.to_csv(output_path)
print(f"\n✅ Saved to {output_path}")


✅ Saved to /home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_positive_reg_pi_on_r_add_scaled.csv


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc

# --- Define sampled parameters ---
gene_params_sampled = [
    "pi_on",            # k_on / (k_on + k_off)
    "k_on",             # directly sampled k_on
    "mrna_half_life",
    "protein_half_life",
    "k_prod_protein",
    "k_prod_mRNA"
]

interaction_params_scaled = [
    "n_gene_1_to_gene_2", "k_add_scaled_gene_1_to_gene_2",
    "n_gene_2_to_gene_1", "k_add_scaled_gene_2_to_gene_1",
    "n_gene_1_to_gene_3", "k_add_scaled_gene_1_to_gene_3",
    "n_gene_2_to_gene_3", "k_add_scaled_gene_2_to_gene_3"
]

param_names = (
    [f"{p}_gene_1" for p in gene_params_sampled] +
    [f"{p}_gene_2" for p in gene_params_sampled] +
    [f"{p}_gene_3" for p in gene_params_sampled] +
    interaction_params_scaled
)

# --- Bounds for sampled parameters ---
param_bounds = {
    # Gene-level
    "pi_on": (0.002, 0.4),                  # activation probability
    "k_on": (0.01, 3),                      # directly sampled
    "mrna_half_life": (0.6, 17),
    "protein_half_life": (7, 200),
    "k_prod_mRNA": (0.2, 60),
    "k_prod_protein": (19, 2700),
    # Interaction-level
    "n_gene_1_to_gene_2": (0.1, 5),
    "n_gene_2_to_gene_1": (0.1, 5),
    "n_gene_1_to_gene_3": (0.1, 5),
    "n_gene_2_to_gene_3": (0.1, 5),
    "k_add_scaled_gene_1_to_gene_2": (0.5, 2),
    "k_add_scaled_gene_2_to_gene_1": (0.5, 2),
    "k_add_scaled_gene_1_to_gene_3": (0.5, 2),
    "k_add_scaled_gene_2_to_gene_3": (0.5, 2),
}

bounds = (
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in interaction_params_scaled]
)

# --- Sampling configuration ---
n_valid_required = 25000
seed = 42

# Latin Hypercube Sampling (log10 for all except pi_on)
log_bounds_lower = [
    np.log10(b[0]) if "pi_on" not in param_names[i] else b[0]
    for i, b in enumerate(bounds)
]
log_bounds_upper = [
    np.log10(b[1]) if "pi_on" not in param_names[i] else b[1]
    for i, b in enumerate(bounds)
]

sampler = qmc.LatinHypercube(d=len(bounds), seed=seed)
sample = sampler.random(n=n_valid_required)

scaled_sample = np.empty_like(sample)
for i, name in enumerate(param_names):
    # Determine log bounds for every parameter (including pi_on)
    lower = np.log10(param_bounds["pi_on"][0]) if "pi_on" in name else log_bounds_lower[i]
    upper = np.log10(param_bounds["pi_on"][1]) if "pi_on" in name else log_bounds_upper[i]

    # Scale in log-space and exponentiate back
    scaled_log = qmc.scale(sample[:, [i]], [lower], [upper]).ravel()
    scaled_sample[:, i] = 10 ** scaled_log



df_sampled = pd.DataFrame(scaled_sample, columns=param_names)

# --- Convert to actual k_off and k_add ---
def convert_params(row):
    converted = {}

    for g in [1, 2, 3]:
        pi = row[f"pi_on_gene_{g}"]
        k_on = row[f"k_on_gene_{g}"]

        # Derive k_off from pi_on and k_on
        k_off = k_on * (1 - pi) / pi

        converted[f"k_on_gene_{g}"] = k_on
        converted[f"k_off_gene_{g}"] = k_off
        converted[f"mrna_half_life_gene_{g}"] = row[f"mrna_half_life_gene_{g}"]
        converted[f"protein_half_life_gene_{g}"] = row[f"protein_half_life_gene_{g}"]
        converted[f"k_prod_mRNA_gene_{g}"] = row[f"k_prod_mRNA_gene_{g}"]
        converted[f"k_prod_protein_gene_{g}"] = row[f"k_prod_protein_gene_{g}"]

    # Convert k_add_scaled → k_add
    for key in row.index:
        if "k_add_scaled" in key:
            tgt_gene = key.split("_")[7]  # target gene index
            k_on_target = converted[f"k_on_gene_{tgt_gene}"]
            converted[key.replace("k_add_scaled", "k_add")] = row[key] * k_on_target
        elif "n_gene" in key:
            converted[key] = row[key]

    return pd.Series(converted)

df_converted = df_sampled.apply(convert_params, axis=1)

# --- Expand to long format ---
rows = []
for idx, row in df_converted.iterrows():
    g1 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_1") and "to" not in k}
    g2 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_2") and "to" not in k}
    g3 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_3") and "to" not in k}
    interactions = {k: v for k, v in row.items() if "gene_" in k and "to" in k}

    rows.append({**g1, **interactions, "pair_id": idx, "gene_id": 1})
    rows.append({**g2, **interactions, "pair_id": idx, "gene_id": 2})
    rows.append({**g3, **interactions, "pair_id": idx, "gene_id": 3})

final_df = pd.DataFrame(rows).reset_index(drop=True)

# --- Save ---
output_path = (
    "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_repression_reg_pi_on_r_add_scaled.csv"
)
final_df.to_csv(output_path)
print(f"\n✅ Saved to {output_path}")


✅ Saved to /home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_repression_reg_pi_on_r_add_scaled.csv


In [1]:
# param_df = pd.read_csv("/home/mzo5929/Keerthana/grnInference/simulation_data/gillespie_simulation/sim_details/lhc_sampled_parameters_negative_reg.csv", index_col = 0)

import numpy as np
import pandas as pd

def hl_to_deg(hl):
    """Convert half-life to degradation rate."""
    return np.log(2) / hl

def compute_steady_state_levels(param_df, gene_id):
    """Compute mean mRNA and protein levels for gene_id (1 or 2), assuming hill = 0.5."""
    assert gene_id in [1, 2], "gene_id must be 1 or 2"

    # Basic parameters
    k_on = param_df["k_on"]
    k_off = param_df["k_off"]
    prod_m = param_df["k_prod_mRNA"]
    prod_p = param_df["k_prod_protein"]
    deg_m = hl_to_deg(param_df["mrna_half_life"])
    deg_p = hl_to_deg(param_df["protein_half_life"])

    # Use .get to safely retrieve interaction term or default to 0
    if gene_id == 2:
        k_add = param_df.get("k_add_gene_1_to_gene_2", 0.0)
    else:
        k_add = param_df.get("k_add_gene_2_to_gene_1", 0.0)

    # Compute effective k_on using hill response = 0.5
    k_on_eff = k_on + 0.5 * k_add
    burst_prob = k_on_eff / (k_on_eff + k_off)

    # Steady-state means
    mean_mRNA = burst_prob * prod_m / deg_m
    mean_protein = mean_mRNA * prod_p / deg_p

    # Store results in DataFrame
    param_df["mean_mRNA_level"] = mean_mRNA
    param_df["mean_protein_level"] = mean_protein

    return param_df

# Usage
param_df = pd.read_csv("/home/mzo5929/Keerthana/grnInference/simulation_data/gillespie_simulation_run_2/sim_details/lhc_sampled_parameters_positive_reg_2.csv", index_col = 0)
param_df = compute_steady_state_levels(param_df, gene_id=2)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Filter out zero or negative values (log scale can't handle them)
values = param_df['mean_protein_level']
values = values[values > 0]

# Define log-spaced bins
n_bins = 100
min_val = values.min()
max_val = values.max()
log_bins = np.logspace(np.log10(min_val), np.log10(max_val), n_bins)

# Plot
plt.figure(figsize=(6, 4))
plt.hist(values, bins=log_bins)
plt.xscale('log')
# plt.yscale('log')
plt.xlabel("Mean protein level (log scale)")
plt.ylabel("Frequency (log scale)")
plt.title("Log-Binned Histogram of Mean Protein Levels")
plt.tight_layout()
plt.show()


In [ ]:
param_df[param_df['mean_mRNA_level'] < 100].shape

In [ ]:
plt.hist(param_df[param_df['mean_mRNA_level'] < 1]['mean_mRNA_level'])